In [1]:
import xarray as xr
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import fnmatch
from pyproj import CRS, Transformer
from pyhdf.SD import SD, SDC 

/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/pyproj/__init__.py:95: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [2]:
def convert_lat_lon_to_x_y(crs, lon, lat):
    transformer = Transformer.from_crs(CRS("+proj=latlon"), crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    return x, y

In [3]:
msg = xr.open_dataset('/mnt/outputs/geoprocessed/20230728151242_msg.nc')
msg

<xarray.Dataset> Size: 30MB
Dimensions:             (y: 410, x: 1235, band: 11, time: 1, band_wavelength: 11)
Coordinates:
  * y                   (y) float64 3kB 3.102e+06 3.105e+06 ... 4.33e+06
  * x                   (x) float64 10kB -9.331e+05 -9.301e+05 ... 2.769e+06
    cloud_mask          (y, x) float64 4MB ...
    latitude            (y, x) float32 2MB ...
    longitude           (y, x) float32 2MB ...
  * band                (band) <U6 264B 'IR_016' 'IR_039' ... 'WV_062' 'WV_073'
  * time                (time) <U19 76B '2023-07-28 15:00:00'
  * band_wavelength     (band_wavelength) float64 88B 1.64 3.92 ... 6.25 7.35
Data variables:
    msg_seviri_fes_3km  int64 8B ...
    Rad                 (band, y, x) float32 22MB ...
Attributes:
    calibration:         radiance
    standard_name:       toa_outgoing_radiance_per_unit_wavenumber
    platform_name:       Meteosat-10
    sensor:              seviri
    units:               mW m-2 sr-1 (cm-1)-1
    orbital_parameters:  {"projection_longitude": 0.0, "projection_latitude":...

In [4]:
def create_fires_ds(matching_modis_files, modis_dir, msg):
    array = np.zeros((msg.y.size, msg.x.size))
    var = "msg_seviri_fes_3km"
    crs_wkt = msg[var].crs_wkt
    crs = CRS(crs_wkt)
    print(os.path.join(modis_dir, matching_modis_files[0]))
    f = SD(os.path.join(modis_dir, matching_modis_files[0]), SDC.READ)
    df = pd.DataFrame()
    sds_obj = f.select('fire mask') # select sds
    fm_data = sds_obj.get() # get sds data

    if fm_data[fm_data > 6].shape[0] > 0:
        for key in ['FP_latitude', 
                        'FP_longitude', 
                        'FP_power']:

            sds_obj = f.select(key) 
            
            data = sds_obj.get()
            df[key] = data.ravel()
    
        for index, row in df.iterrows():
            lon = row['FP_longitude']
            lat = row['FP_latitude']
            x_sel, y_sel = convert_lat_lon_to_x_y(crs, lon, lat)
            selected = msg.sel(x=x_sel, y=y_sel, method='nearest')
            # Get the indices of the nearest point
            x_idx = msg.get_index('x').get_loc(selected['x'].item())
            y_idx = msg.get_index('y').get_loc(selected['y'].item())
            array[y_idx, x_idx] = 1
        da = xr.DataArray(
            array,
            coords={"y": msg.y, "x": msg.x},
            dims=("y", "x")
    )
    else:
        print('No fires detected')
        return None
    return da

In [5]:
csv_file = '/mnt/data8tb/fire_detection/msg-MOD021KM-timestamps_2023-06-01_2023-09-30.csv'
modis_dir = '/mnt/data8tb/fire_detection/modis/MOD/AF'
msg_dir = '/mnt/outputs/geoprocessed'
labels_path = '/mnt/data8tb/fire_detection/modis/MOD/geoprocessed_fires'

In [6]:
pairs = pd.read_csv(csv_file)

In [10]:
for index, row in pairs.iterrows():
    modis_time = datetime.strptime(row.MODIS, "%Y-%m-%d %H:%M:%S")
    year = modis_time.strftime("%Y")
    doy = modis_time.strftime("%j")  # Day of year
    hhmm = modis_time.strftime("%H%M")
    filename_pattern = f"MOD14.A{year}{doy}.{hhmm}.061.*.hdf"
    matching_modis_files = [f for f in os.listdir(modis_dir) if fnmatch.fnmatch(f, filename_pattern)]
    if len(matching_modis_files) == 0:
        print('No MODIS file found for timestamp', modis_time)
        continue
    msg_time = datetime.strptime(row.MSG, "%Y-%m-%d %H:%M:%S")
    msg_filename = msg_time.strftime("%Y%m%d%H%M%S_msg.nc")
    modis_filename = msg_time.strftime("%Y%m%d%H%M%S_af.nc")
    if os.path.exists(os.path.join(msg_dir, msg_filename)):
        ds = xr.open_dataset(os.path.join(msg_dir, msg_filename))
        ds_fires = create_fires_ds(matching_modis_files, modis_dir, ds)
        if ds_fires is None:
            continue
        ds_fires.attrs["modis_datetime"] = str(modis_time)
        ds_fires = ds_fires.to_dataset(name="fires")
        ds_fires.to_netcdf(os.path.join(labels_path, modis_filename))
    else:
        print('No MSG file found for timestamp', msg_time)

/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.0710.061.2023152152810.hdf
/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.0715.061.2023152152901.hdf
/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.0845.061.2023152152900.hdf
/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.0850.061.2023152152912.hdf
/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.0855.061.2023152152933.hdf
/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.1025.061.2023152152908.hdf
/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.1030.061.2023152152815.hdf
/mnt/data8tb/fire_detection/modis/MOD/AF/MOD14.A2023152.1205.061.2023152211246.hdf
No fires detected
No MODIS file found for timestamp 2023-06-01 18:15:00
No MODIS file found for timestamp 2023-06-01 18:20:00
No MODIS file found for timestamp 2023-06-01 19:50:00
No MODIS file found for timestamp 2023-06-01 19:55:00
No MODIS file found for timestamp 2023-06-01 21:30:00
No MODIS file found for timestamp 2023-06-01 21:

In [11]:
os.path.join(labels_path, modis_filename)

'/mnt/data8tb/fire_detection/modis/MOD/geoprocessed_fires/20230930111241_af.nc'

## Check af files

In [15]:
for file in os.listdir(labels_path):
    af_file = xr.open_dataset(os.path.join(labels_path, file))
    #count the number of fires by countinng how many 1 in the array
    count = np.count_nonzero(af_file.fires.values)
    print(f'The number of fires detected in {file} is {count}')

The number of fires detected in 20230826101242_af.nc is 36
The number of fires detected in 20230813092742_af.nc is 1
The number of fires detected in 20230810091242_af.nc is 4
The number of fires detected in 20230902064242_af.nc is 1
The number of fires detected in 20230819084241_af.nc is 1
The number of fires detected in 20230926081242_af.nc is 159
The number of fires detected in 20230807084242_af.nc is 6
The number of fires detected in 20230828062742_af.nc is 3
The number of fires detected in 20230607094241_af.nc is 5
The number of fires detected in 20230902082742_af.nc is 13
The number of fires detected in 20230828081241_af.nc is 1
The number of fires detected in 20230930111241_af.nc is 1
The number of fires detected in 20230822085742_af.nc is 93
The number of fires detected in 20230901105742_af.nc is 4
The number of fires detected in 20230820074242_af.nc is 23
The number of fires detected in 20230903085742_af.nc is 17
The number of fires detected in 20230814101241_af.nc is 2
The num

## Check af patches

In [13]:
import os
import rioxarray
import numpy as np
patches_path = '/mnt/data8tb/fire_detection/modis/MOD/patched'
for file in os.listdir(patches_path):
    af_file = rioxarray.open_rasterio(os.path.join(patches_path, file))
    #count the number of fires by countinng how many 1 in the array
    count = np.count_nonzero(af_file.values)
    if count > 10:
        print(f'The number of fires detected in {file} is {count}')

The number of fires detected in 20230926081242_patch_454.tif is 14
The number of fires detected in 20230823081242_patch_334.tif is 13
The number of fires detected in 20230808074242_patch_455.tif is 17
The number of fires detected in 20230926081242_patch_416.tif is 18
The number of fires detected in 20230724104241_patch_204.tif is 22
The number of fires detected in 20230725094242_patch_249.tif is 29
The number of fires detected in 20230724104241_patch_205.tif is 11
The number of fires detected in 20230822085742_patch_334.tif is 32
The number of fires detected in 20230824084242_patch_334.tif is 13
The number of fires detected in 20230928075742_patch_454.tif is 18
The number of fires detected in 20230921095742_patch_249.tif is 19
The number of fires detected in 20230615084242_patch_301.tif is 14
The number of fires detected in 20230801074241_patch_455.tif is 17
The number of fires detected in 20230724104241_patch_203.tif is 23
The number of fires detected in 20230724104241_patch_202.tif i

## Correct geoprocessed fires files 
I will try to clear the fires by intersection with msg and cut the af that in msg band2 have less than 2

In [89]:
fires_path = '/mnt/data8tb/fire_detection/modis/MOD/geoprocessed_fires'
msg_path = '/mnt/outputs/geoprocessed'
output_fires_path = '/mnt/data8tb/fire_detection/modis/MOD/geoprocessed_fires_edited/'

In [93]:
for file in os.listdir(fires_path):
    fires = xr.open_dataset(os.path.join(fires_path, file))
    print(f'fires detected in {file}, {np.count_nonzero(fires.fires.values)}')
    msg_file = file.split('_')[0]+'_msg.nc'
    msg = xr.open_dataset(os.path.join(msg_path, msg_file))
    y_s = np.where(fires.fires.values==1)[0]
    x_s = np.where(fires.fires.values==1)[1]
    for x_ind, y_ind in zip(x_s, y_s):
        pixel = msg.isel(x=x_ind, y=y_ind).Rad[1]
        x, y = pixel.x.values, pixel.y.values
        thermal_band = pixel.values
        if thermal_band<=2:
            fires.loc[dict(x=x, y=y)] = 0
    print(f'fires remained in {file}, {np.count_nonzero(fires.fires.values)}')
    if np.count_nonzero(fires.fires.values) > 0:
        fires.to_netcdf(os.path.join(output_fires_path, file))

fires detected in 20230826101242_af.nc, 36
fires remained in 20230826101242_af.nc, 0
fires detected in 20230813092742_af.nc, 1
fires remained in 20230813092742_af.nc, 0
fires detected in 20230810091242_af.nc, 4
fires remained in 20230810091242_af.nc, 0
fires detected in 20230902064242_af.nc, 1
fires remained in 20230902064242_af.nc, 0
fires detected in 20230819084241_af.nc, 1
fires remained in 20230819084241_af.nc, 0
fires detected in 20230926081242_af.nc, 159
fires remained in 20230926081242_af.nc, 0
fires detected in 20230807084242_af.nc, 6
fires remained in 20230807084242_af.nc, 0
fires detected in 20230828062742_af.nc, 3
fires remained in 20230828062742_af.nc, 0
fires detected in 20230607094241_af.nc, 5
fires remained in 20230607094241_af.nc, 0
fires detected in 20230902082742_af.nc, 13
fires remained in 20230902082742_af.nc, 0
fires detected in 20230828081241_af.nc, 1
fires remained in 20230828081241_af.nc, 0
fires detected in 20230930111241_af.nc, 1
fires remained in 202309301112

In [94]:
for file in os.listdir(output_fires_path):
    af_file = xr.open_dataset(os.path.join(output_fires_path, file))
    #count the number of fires by countinng how many 1 in the array
    count = np.count_nonzero(af_file.fires.values)
    print(f'The number of fires detected in {file} is {count}')

The number of fires detected in 20230822085742_af.nc is 31
The number of fires detected in 20230614094241_af.nc is 1
The number of fires detected in 20230727092742_af.nc is 1
The number of fires detected in 20230807101242_af.nc is 6
The number of fires detected in 20230725094242_af.nc is 26
The number of fires detected in 20230917102741_af.nc is 1
The number of fires detected in 20230830092741_af.nc is 1
The number of fires detected in 20230723095742_af.nc is 4
The number of fires detected in 20230720111242_af.nc is 1
The number of fires detected in 20230820091242_af.nc is 3
The number of fires detected in 20230821095741_af.nc is 1
The number of fires detected in 20230825111242_af.nc is 2
The number of fires detected in 20230723081242_af.nc is 9
The number of fires detected in 20230717104242_af.nc is 5
The number of fires detected in 20230921095742_af.nc is 3
The number of fires detected in 20230824084242_af.nc is 2
The number of fires detected in 20230821082741_af.nc is 3
The number o